# Assignment 2: Mobile Money Fraud Detection

## Class Imbalance Handling and Hyperparameter Tuning

This notebook covers:

1. Dataset and problem description  
2. Class imbalance analysis  
3. Baseline modelling  
4. Class balancing techniques  
5. Hyperparameter tuning  
6. Model evaluation and comparison  
7. Discussion and conclusion  

**Important:** Class balancing is applied only to the training data to avoid data leakage.

In [1]:
!pip install imbalanced-learn xgboost seaborn

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.63.0-cp313-cp313-win_amd64.whl.metadata (121 kB)
  Using cached kiwisolver-1.5.0-cp313-cp313-win_amd64.whl.metadata (5.2 kB)
  Using cached pillow-12.2.0-cp313-cp313-win_amd64.whl.metadata (9.0 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
    --------------------------------------- 0.3/12.4 MB ? eta -:--:--
   - ------------------------------------


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Import Required Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost is not installed. GradientBoostingClassifier will be used as an alternative.")

Matplotlib is building the font cache; this may take a moment.


## 2. Load Dataset

In [ ]:
DATA_PATH = "synthetic_mobile_money_transaction_dataset.csv"

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()

print("Dataset loaded successfully")
print("Dataset shape:", df.shape)
display(df.head())

## 3. Data Understanding

In [ ]:
print("Dataset information")
df.info()

print("\nSummary statistics")
display(df.describe(include="all").T)

print("\nMissing values")
display(df.isnull().sum())

print("\nDuplicate records")
print(df.duplicated().sum())

## 4. Task 1: Dataset and Problem Description

In [ ]:
target_col = "isFraud"

if target_col not in df.columns:
    raise ValueError("Target column isFraud was not found. Check the dataset columns.")

records = df.shape[0]
variables = df.shape[1]

print("Number of records:", records)
print("Number of variables:", variables)

class_counts = df[target_col].value_counts().sort_index()
class_percentages = df[target_col].value_counts(normalize=True).sort_index() * 100

class_summary = pd.DataFrame({
    "Class": class_counts.index,
    "Count": class_counts.values,
    "Percentage": class_percentages.values
})

display(class_summary)

In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(x=target_col, data=df)
plt.title("Fraud vs Non Fraud Transactions")
plt.xlabel("isFraud")
plt.ylabel("Count")
plt.show()

### Problem Description

The problem is a binary classification task. The aim is to classify each mobile money transaction as either fraudulent or legitimate using transaction characteristics such as amount, transaction type, sender balance and recipient balance.

## 5. Task 2: Class Imbalance Analysis

In [ ]:
fraud_count = class_counts.get(1, 0)
non_fraud_count = class_counts.get(0, 0)

fraud_percentage = class_percentages.get(1, 0)
non_fraud_percentage = class_percentages.get(0, 0)

print(f"Non fraud transactions: {non_fraud_count:,} ({non_fraud_percentage:.2f}%)")
print(f"Fraud transactions: {fraud_count:,} ({fraud_percentage:.2f}%)")

if fraud_count > 0:
    print(f"Imbalance ratio: {non_fraud_count / fraud_count:.2f}:1")

### Why Accuracy May Be Misleading

Accuracy can be misleading in fraud detection because the majority class is usually non fraud. A model may achieve high accuracy by predicting most transactions as non fraud, but still fail to identify fraudulent transactions. Therefore, Precision, Recall, F1 Score, ROC AUC and Precision Recall AUC are more useful.

## 6. Exploratory Data Analysis

In [ ]:
if "amount" in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df["amount"], bins=50)
    plt.title("Transaction Amount Distribution")
    plt.xlabel("Amount")
    plt.ylabel("Count")
    plt.show()

possible_type_columns = ["type", "transactionType", "transaction_type"]
type_col = None

for col in possible_type_columns:
    if col in df.columns:
        type_col = col
        break

if type_col is not None:
    plt.figure(figsize=(8, 5))
    sns.countplot(x=type_col, hue=target_col, data=df)
    plt.title("Transaction Type by Fraud Status")
    plt.xticks(rotation=45)
    plt.xlabel("Transaction Type")
    plt.ylabel("Count")
    plt.show()
else:
    print("No transaction type column found.")

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])

plt.figure(figsize=(10, 7))
sns.heatmap(numeric_df.corr(), cmap="coolwarm", annot=False)
plt.title("Correlation Heatmap")
plt.show()

## 7. Feature Preparation

In [ ]:
df_clean = df.drop_duplicates().copy()

X = df_clean.drop(columns=[target_col])
y = df_clean[target_col]

# Drop high cardinality object identifiers because they may not generalise well
high_cardinality_cols = []

for col in X.columns:
    if X[col].dtype == "object" and X[col].nunique() > 100:
        high_cardinality_cols.append(col)

print("High cardinality columns dropped:", high_cardinality_cols)
X = X.drop(columns=high_cardinality_cols, errors="ignore")

print("Predictor shape:", X.shape)

## 8. Feature Engineering

In [ ]:
X_fe = X.copy()

if "oldBalInitiator" in X_fe.columns and "newBalInitiator" in X_fe.columns:
    X_fe["initiator_balance_difference"] = X_fe["oldBalInitiator"] - X_fe["newBalInitiator"]

if "oldbalanceOrg" in X_fe.columns and "newbalanceOrig" in X_fe.columns:
    X_fe["initiator_balance_difference"] = X_fe["oldbalanceOrg"] - X_fe["newbalanceOrig"]

if "oldBalRecipient" in X_fe.columns and "newBalRecipient" in X_fe.columns:
    X_fe["recipient_balance_difference"] = X_fe["oldBalRecipient"] - X_fe["newBalRecipient"]

if "oldbalanceDest" in X_fe.columns and "newbalanceDest" in X_fe.columns:
    X_fe["recipient_balance_difference"] = X_fe["oldbalanceDest"] - X_fe["newbalanceDest"]

numeric_features = X_fe.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_fe.select_dtypes(exclude=[np.number]).columns.tolist()

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)
display(X_fe.head())

## 9. Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_fe,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

print("\nTraining class distribution")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting class distribution")
print(y_test.value_counts(normalize=True) * 100)

## 10. Preprocessing Pipeline

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

## 11. Evaluation Function

In [ ]:
results = []

def evaluate_model(model_name, model, X_test, y_test):
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        y_proba = model.decision_function(X_test)
    else:
        y_proba = y_pred

    result = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC AUC": roc_auc_score(y_test, y_proba),
        "PR AUC": average_precision_score(y_test, y_proba)
    }

    results.append(result)

    print("\n" + "="*80)
    print(model_name)
    print("="*80)
    print(classification_report(y_test, y_pred, zero_division=0))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("ROC AUC:", result["ROC AUC"])
    print("PR AUC:", result["PR AUC"])

    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues")
    plt.title(f"Confusion Matrix: {model_name}")
    plt.show()

    RocCurveDisplay.from_predictions(y_test, y_proba)
    plt.title(f"ROC Curve: {model_name}")
    plt.show()

    PrecisionRecallDisplay.from_predictions(y_test, y_proba)
    plt.title(f"Precision Recall Curve: {model_name}")
    plt.show()

## 12. Task 3: Baseline Modelling

In [ ]:
baseline_lr = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

baseline_rf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])

baseline_lr.fit(X_train, y_train)
evaluate_model("Baseline Logistic Regression", baseline_lr, X_test, y_test)

baseline_rf.fit(X_train, y_train)
evaluate_model("Baseline Random Forest", baseline_rf, X_test, y_test)

## 13. Task 4: Class Balancing Technique 1: Class Weighting

In [ ]:
weighted_lr = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
])

weighted_rf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

weighted_lr.fit(X_train, y_train)
evaluate_model("Weighted Logistic Regression", weighted_lr, X_test, y_test)

weighted_rf.fit(X_train, y_train)
evaluate_model("Weighted Random Forest", weighted_rf, X_test, y_test)

## 14. Task 4: Class Balancing Technique 2: SMOTE

In [ ]:
smote_lr = ImbPipeline(steps=[
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

smote_rf = ImbPipeline(steps=[
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("model", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])

smote_lr.fit(X_train, y_train)
evaluate_model("SMOTE Logistic Regression", smote_lr, X_test, y_test)

smote_rf.fit(X_train, y_train)
evaluate_model("SMOTE Random Forest", smote_rf, X_test, y_test)

## 15. Optional Additional Balancing: Random Undersampling

In [ ]:
undersample_rf = ImbPipeline(steps=[
    ("preprocessor", preprocessor),
    ("undersample", RandomUnderSampler(random_state=42)),
    ("model", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])

undersample_rf.fit(X_train, y_train)
evaluate_model("Random Undersampling Random Forest", undersample_rf, X_test, y_test)

## 16. Task 5: Hyperparameter Tuning

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

### 16.1 Random Forest Tuning

In [ ]:
rf_tuning_pipeline = ImbPipeline(steps=[
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("model", RandomForestClassifier(random_state=42, n_jobs=-1))
])

rf_param_dist = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2", None]
}

rf_search = RandomizedSearchCV(
    estimator=rf_tuning_pipeline,
    param_distributions=rf_param_dist,
    n_iter=10,
    scoring="f1",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=2
)

rf_search.fit(X_train, y_train)

print("Best Random Forest parameters:")
print(rf_search.best_params_)
print("Best CV F1 score:")
print(rf_search.best_score_)

best_rf = rf_search.best_estimator_
evaluate_model("Tuned SMOTE Random Forest", best_rf, X_test, y_test)

### 16.2 XGBoost or Gradient Boosting Tuning

In [ ]:
if XGBOOST_AVAILABLE:
    xgb_model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    xgb_tuning_pipeline = ImbPipeline(steps=[
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=42)),
        ("model", xgb_model)
    ])

    xgb_param_dist = {
        "model__n_estimators": [100, 200, 300],
        "model__max_depth": [3, 5, 7],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__subsample": [0.8, 1.0],
        "model__colsample_bytree": [0.8, 1.0]
    }

    xgb_search = RandomizedSearchCV(
        estimator=xgb_tuning_pipeline,
        param_distributions=xgb_param_dist,
        n_iter=10,
        scoring="f1",
        cv=cv,
        random_state=42,
        n_jobs=-1,
        verbose=2
    )

    xgb_search.fit(X_train, y_train)

    print("Best XGBoost parameters:")
    print(xgb_search.best_params_)
    print("Best CV F1 score:")
    print(xgb_search.best_score_)

    best_xgb = xgb_search.best_estimator_
    evaluate_model("Tuned SMOTE XGBoost", best_xgb, X_test, y_test)

else:
    gb_tuning_pipeline = ImbPipeline(steps=[
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=42)),
        ("model", GradientBoostingClassifier(random_state=42))
    ])

    gb_param_dist = {
        "model__n_estimators": [100, 200],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__max_depth": [3, 5],
        "model__subsample": [0.8, 1.0]
    }

    gb_search = RandomizedSearchCV(
        estimator=gb_tuning_pipeline,
        param_distributions=gb_param_dist,
        n_iter=8,
        scoring="f1",
        cv=cv,
        random_state=42,
        n_jobs=-1,
        verbose=2
    )

    gb_search.fit(X_train, y_train)

    print("Best Gradient Boosting parameters:")
    print(gb_search.best_params_)
    print("Best CV F1 score:")
    print(gb_search.best_score_)

    best_gb = gb_search.best_estimator_
    evaluate_model("Tuned SMOTE Gradient Boosting", best_gb, X_test, y_test)

## 17. Task 6: Model Evaluation and Comparison

In [ ]:
results_df = pd.DataFrame(results).sort_values(by="F1 Score", ascending=False)
display(results_df)

results_df.to_csv("assignment2_model_comparison_results.csv", index=False)

metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1 Score", "ROC AUC", "PR AUC"]

for metric in metrics_to_plot:
    plt.figure(figsize=(12, 6))
    sns.barplot(data=results_df, x="Model", y=metric)
    plt.title(f"Model Comparison by {metric}")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()